# 🦺 Industrial Safety Vision: Dataset & Pipeline Exploration

Welcome to the **Industrial Safety Vision** exploration notebook. This notebook walks through:
1. **Understanding Dataset Formats**: Inspecting YOLO labels (`helmet`, `vest`, `person`, `forklift`).
2. **Geometric Calculations**: Evaluating bounding box Intersection-over-Union (IoU) and PPE overlap.
3. **Danger Zone Definition**: Checking polygon point-in-polygon containment for hazard zones.
4. **End-to-End Pipeline**: Running detection, IoU tracking, and temporal safety rule smoothing.

## 1. Imports and Setup

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in python path
repo_root = Path("..").resolve()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from industrial_safety_vision.core import BoundingBox, Detection
from industrial_safety_vision.tracking.tracker import SimpleIoUTracker
from industrial_safety_vision.safety.danger_zone import DangerZone, track_in_zone
from industrial_safety_vision.safety.rules import SafetyRulesEngine, SafetyRulesConfig
from industrial_safety_vision.inference.mock_detector import MockSafetyDetector
from industrial_safety_vision.demo.synthetic import generate_synthetic_frame
from industrial_safety_vision.utils.geometry import bbox_iou, bbox_overlap_ratio

print("✅ Industrial Safety Vision modules loaded successfully.")

## 2. Dataset Structure and Target Classes

In typical construction and industrial safety datasets (e.g. `LibreYOLO/construction-safety-gsnvb`), annotations follow the YOLO format: `<class_id> <x_center> <y_center> <width> <height>` (normalized [0, 1]).

Common target classes:
- `0`: `helmet`
- `1`: `no-helmet`
- `2`: `no-vest`
- `3`: `person`
- `4`: `vest`

In [ ]:
class_labels = {0: "helmet", 1: "no-helmet", 2: "no-vest", 3: "person", 4: "vest"}
print("Target Classes:")
for cid, cname in class_labels.items():
    print(f"  [{cid}] {cname}")

## 3. Geometric Reasoning: PPE Overlap and IoU

A worker is considered compliant if the detected PPE (`helmet`, `vest`) has sufficient spatial overlap with the tracked `person` bounding box.

In [ ]:
# Worker bbox
worker_box = BoundingBox(x1=100, y1=100, x2=250, y2=450)
# Helmet bbox located near worker head
helmet_box = BoundingBox(x1=140, y1=90, x2=210, y2=150)

overlap = bbox_overlap_ratio(parent=worker_box, child=helmet_box)
iou = bbox_iou(worker_box, helmet_box)

print(f"Worker Box Area: {worker_box.area} px")
print(f"Helmet Box Area: {helmet_box.area} px")
print(f"PPE-in-Person Overlap Ratio: {overlap:.2%}")
print(f"Intersection-over-Union (IoU): {iou:.3f}")

## 4. Danger Zone Spatial Containment

Danger zones are defined as arbitrary 2D polygons on the factory or construction site floor. A worker's position is estimated using their **bottom-center point** (feet/ground contact).

In [ ]:
# Define hazardous loading area polygon
hazard_zone = DangerZone(
    name="Forklift_Loading_Dock",
    polygon=[(200, 300), (500, 300), (520, 460), (180, 460)]
)

# Track outside hazard zone
safe_worker = BoundingBox(x1=50, y1=100, x2=150, y2=250)
# Track inside hazard zone (bottom center inside polygon)
danger_worker = BoundingBox(x1=300, y1=200, x2=400, y2=400)

from industrial_safety_vision.tracking.track_types import Track
track_safe = Track(track_id=1, class_name="person", bbox=safe_worker, confidence=0.95)
track_danger = Track(track_id=2, class_name="person", bbox=danger_worker, confidence=0.98)

print(f"Worker #1 in danger zone: {track_in_zone(track_safe, hazard_zone)}")
print(f"Worker #2 in danger zone: {track_in_zone(track_danger, hazard_zone)}")

## 5. Visualizing Sample Frames & Safety Zones

In [ ]:
# Generate a synthetic test scene
frame = generate_synthetic_frame(width=640, height=360, frame_index=15, annotated=True)

plt.figure(figsize=(10, 6))
plt.imshow(frame)
plt.title("Synthetic Industrial Safety Scene with Worker, PPE & Hazard Polygon", fontsize=12)
plt.axis("off")
plt.show()

## 6. Running the Safety Rules Engine

The `SafetyRulesEngine` applies **temporal smoothing** and **cooldown periods** to prevent false alarms and alert fatigue.

In [ ]:
engine = SafetyRulesEngine()
print("Safety Engine Initialized with Configured Rules:")
print(f"- Missing Helmet Rule enabled: {engine.config.missing_helmet.enabled}")
print(f"- Missing Vest Rule enabled: {engine.config.missing_vest.enabled}")
print(f"- Danger Zone Rule enabled: {engine.config.danger_zone.enabled}")
print(f"- Vehicle Proximity Rule enabled: {engine.config.vehicle_proximity.enabled}")